# 🛡️ Tank duel — PPO + GRU self-play on Colab
Train here (GPU for the network, all vCPUs for the arenas), duel on your laptop with `referee.py`.
Runtime → *Change runtime type* → **A100 GPU**. Run the cells top to bottom; after a disconnect run them again — training **resumes** from Google Drive.

In [ ]:
import os, multiprocessing
print("vCPU:", multiprocessing.cpu_count())
!nvidia-smi -L

## 1. Drive (checkpoints survive disconnects) and code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # only for checkpoints/exports (they survive disconnects)

RUN_DIR = '/content/drive/MyDrive/tank_runs/run2'   # NEW name = fresh run; the SAME name = --resume continues it
ARENA_URL = 'https://github.com/BohatyrovIvanUcu/tank_battle_playground.git'   # shared arena + referee
ARENA_REF = 'main'                                   # or the rules tag, e.g. 'rules-v1.1'
TRAIN_URL = 'https://github.com/partum55/tank-game.git'   # PPO + GRU trainer
os.makedirs(RUN_DIR, exist_ok=True)

In [ ]:
%cd /content
!rm -rf playground tank-game
!git clone -q {ARENA_URL} playground
!git clone -q {TRAIN_URL} tank-game
%cd /content/playground
!git checkout -q {ARENA_REF} && echo "arena:" && git log --oneline | head -1
!cd /content/tank-game && echo "trainer:" && git log --oneline | head -1

## 2. Build the Rust arena (headless: no window libraries needed)

In [ ]:
!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ['PATH'] += ':/root/.cargo/bin'
!pip install -q maturin
!maturin build --release --no-default-features -q -o /content/wheels
# --no-deps: Colab already has numpy/torch; letting pip touch numpy breaks them ("numpy.dtype size changed")
!pip install -q --no-deps --force-reinstall /content/wheels/tank_arena-*.whl

In [ ]:
os.environ['PYTHONPATH'] = '/content/playground/python:/content/tank-game'
%cd /content/tank-game
!python -c "import numpy, torch, tank_arena, tank_env; print('numpy', numpy.__version__, '| torch', torch.__version__, '| rules', tank_env.RULES_VERSION)"
!python train_ppo.py --self-check

## 3. Train
On a GPU runtime the arenas run **on the GPU** (`--env gpu` is chosen automatically: 2048 games in parallel, the Rust engine only builds the maps and does the evaluation). Without a GPU it falls back to the Rust engine on the CPU workers.
Log columns: `win/loss/draw` = learner vs hall-of-fame + bot over the last updates, `bot win` = vs the rule-based anchor, `EVAL vs_bot / vs_old` = fixed maps on the **Rust** engine, both sides (1.0 = always wins, 0.5 = even), `*best` = new best export. Watch `sps`: it should be tens of thousands or more on the GPU env.
Raise `--steps` (e.g. `300e6`) for a longer run; continuing a run with `--resume` and the same `RUN_DIR` is safe.

**GPU sizing is automatic**: the trainer benchmarks 1k, 2k, 4k ... arenas, finds where the GPU stops getting faster, and then caps the number of arenas so the run keeps >= 400 PPO updates (PPO learns per update - a giant batch with only a handful of updates learns *worse*). So a bigger `--steps` budget = more arenas = more of the GPU used. The first lines of the log show the benchmark table and the choice; the `vram` column shows memory use. Force a size with `--gpu-envs N`.

**Measured on an A100:** a simulator step takes ~37 ms whether it runs 1k or 65k arenas (it is launch-bound, the GPU is mostly idle), so more arenas are almost free - what limits learning is the number of PPO *updates*. Hence a short rollout and many arenas: `--rollout 32 --steps 600e6` -> ~32k arenas, ~570 updates, roughly 35-45 min. Without evaluations an update took 3.0 s (rollout 2.4 s + PPO 0.6 s); evaluations on the Rust engine now run in parallel on the idle vCPUs.
Arenas are a mix of the official maps and extra training-only layouts (`--map-mix 0.3`); evaluation always uses the official Rust maps.
**Continuing a run (`--resume`, same `RUN_DIR`) is safe, also with different `--steps`/`--rollout`/arena counts.** A new `RUN_DIR` starts from scratch.

**CUDA graph (new, `--cuda-graph auto`):** the whole rollout step is recorded once and replayed with a single launch, which removes the CPU launch bottleneck. At start the log prints one line, e.g. `rollout: CUDA graph ON: 37.0 ms/step eager -> 3.1 ms/step replayed (11.9x)` - it times both modes itself and falls back to the plain loop (`CUDA graph OFF (capture failed: ...)`) if anything goes wrong. If the run dies with a CUDA error right after the capture, restart the training cell with `--cuda-graph off` and send me the message.

In [ ]:
%cd /content/tank-game
!python train_ppo.py --save-dir {RUN_DIR} --steps 600e6 --rollout 32 --resume

## 4. Hand-off: one zip to download
Training keeps `RUN_DIR/bundle/` fresh at every evaluation (~15 MB: config, per-update metrics, the best and latest agents, a few hall-of-fame snapshots).
Run the cell below (also fine while a session is alive after an interrupt) and send the downloaded **`tank_bundle.zip`**; assess it with

```bash
cd tank-game && PYTHONPATH=../playground/python:. python3 assess.py tank_bundle.zip
```
The verdict says whether the agent is still improving (keep training with `--resume`), plateaued, or stuck. The agent itself is `export_best/` inside the zip (NumPy only, duel it with the shared `referee.py`).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/tank_bundle', 'zip', RUN_DIR + '/bundle')
files.download('/content/tank_bundle.zip')